# Übung 6: Fortgeschrittenes maschinelles Lernen

In dieser Übung untersuchen wir fortgeschrittene Methoden des maschinellen Lernens mit TensorFlow und Keras. Mit dem Datensatz `colorectal_histology` erstellen und bewerten wir Deep-Learning-Modelle. Das Notebook umfasst folgende Schritte:

1. **Datensatz vorbereiten:** Daten laden und vorverarbeiten, einschließlich Datenerweiterung (Data Augmentation).
2. **Modell trainieren:** MobileNetV2 mithilfe von Transfer Learning trainieren und bewerten.
3. **Klassenungleichgewicht untersuchen:** Klassen-Gewichte und Focal Loss vergleichen.
4. **Qualitative Analyse:** Modellvorhersagen mit Grad-CAM visualisieren und kritisch betrachten.

Am Ende könnt ihr besser erklären, wie sich unausgewogene Datensätze auf Modelle auswirken und warum mehrere Kennzahlen nötig sind.

> **Wichtig:** Die Klassen sind Labels eines Lehrdatensatzes. Dieses Modell ist kein medizinisches Diagnosewerkzeug; die Ergebnisse sind keine klinische Evidenz.

## Bibliotheken importieren

In [ ]:
import tensorflow.keras as K
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Lambda
from tensorflow.keras.models import Model

# Funktioniert sowohl bei Start aus dem Projektstamm als auch aus notebooks/.
from pathlib import Path
import sys
NOTEBOOK_ORDNER = Path.cwd() / 'notebooks' if (Path.cwd() / 'notebooks').exists() else Path.cwd()
if str(NOTEBOOK_ORDNER) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_ORDNER))

from util import *
import random
import sklearn

# Module automatisch neu laden
%load_ext autoreload
%autoreload 2

# Zufallsstartwerte festlegen
tf.random.set_seed(313)
np.random.seed(313)
random.seed(313)
sklearn.set_config(assume_finite=True)

# Hyperparameter und Konstanten
subset_frac = 0.5
batch_size = 64
learning_rate = 0.001
test_size = 500
num_unfrozen_layers = 3
epochs = 20

## Datensatz vorbereiten

Wir verwenden den Datensatz `colorectal_histology`. Er enthält Bilder histologischer Gewebeschnitte und acht Klassen-Labels:

1. **Tumor:** Epithelgewebe mit einem malignen Tumor
2. **Stroma:** Bindegewebe ohne besondere Merkmale
3. **Complex:** Gewebe mit malignen und benignen Merkmalen
4. **Lympho:** Gewebe mit Lymphozyten
5. **Debris:** Hintergrund und Gewebereste
6. **Mucosa:** Schleimhautgewebe
7. **Adipose:** Fettgewebe
8. **Empty:** Kein Gewebe

Wir laden den Datensatz und bereiten ihn für das Training vor. Dazu teilen wir ihn in Trainings- und Testdaten und wenden auf die Trainingsdaten eine Datenerweiterung an. Mit `subset_frac` wählen wir nur einen Anteil aus, damit der ausgewogene und der unausgewogene Fall vergleichbare Größen haben.

In [ ]:
# Datensatz colorectal_histology laden
dataset, info = tfds.load('colorectal_histology', with_info=True)
label_names = info.features['label'].names
tfds.show_examples(dataset['train'], info)

# Bilder und Labels in Arrays aufteilen 
X = np.array([example['image'] for example in tfds.as_numpy(dataset['train'])])
Y = np.array([example['label'] for example in tfds.as_numpy(dataset['train'])])

num_classes = info.features['label'].num_classes

# Aufteilung in Trainings- und Testdaten (80 % / 20 %)
X, x_test, Y, y_test = train_test_split(X, Y, test_size=test_size, stratify=Y)

# Zufällig den Anteil subset_frac auswählen
x_train, _, y_train, _ = train_test_split(X, Y, train_size=subset_frac)

# Klassenverteilungen von Training und Test darstellen
plot_label_distribution(y_train)
plot_label_distribution(y_test)

print("Anzahl Trainingsbeispiele:", len(x_train))
print("Anzahl Testbeispiele:", len(x_test))

Für diese Aufgabe verwenden wir MobileNetV2. Dieses kompakte Deep-Learning-Modell eignet sich für mobile Geräte und eingebettete Systeme. Wir nutzen das vortrainierte MobileNetV2 aus TensorFlow Keras Applications. Beim **Transfer Learning** wird ein vortrainiertes Modell übernommen und anschließend an den eigenen Datensatz angepasst. Deshalb müssen die Bilder passend zu MobileNetV2 vorverarbeitet werden. Dafür steht `preprocess_input` bereit. Zusätzlich codieren wir die Labels mit One-Hot-Encoding.

In [ ]:
def preprocess_data(X, Y, num_classes=8):
    X_p = # TODO: Bilder vorverarbeiten
    Y_p = # TODO: Labels mit One-Hot-Encoding codieren
    return X_p, Y_p

def data_augmentation(X):
    datagen = K.preprocessing.image.ImageDataGenerator(
        featurewise_center=False,  # Set input mean to 0 over the dataset
        featurewise_std_normalization=False,  # Divide inputs by std of the dataset
        rotation_range=10,  # Degree range for random rotations
        width_shift_range=0.1,  # Range for random horizontal shifts
        height_shift_range=0.1,  # Range for random vertical shifts
        horizontal_flip=True,  # Randomly flip inputs horizontally
        zoom_range=0.2,  # Range for random zoom
        brightness_range=[0.8, 1.2],  # Range for picking a brightness shift value
    )
    datagen.fit(X)
    return datagen

# Daten vorverarbeiten
x_train, y_train = preprocess_data(x_train, y_train)
x_test, y_test = preprocess_data(x_test, y_test)

# Datengenerator für die Datenerweiterung erstellen
datagen = data_augmentation(x_train)

## MobileNetV2-Modell mit dem Datensatz trainieren

Zur Bewertung eines Modells gibt es verschiedene quantitative Kennzahlen. In früheren Übungen haben wir die Genauigkeit (**Accuracy**) verwendet. Bei unausgewogenen Datensätzen kann sie jedoch irreführend sein. Deshalb betrachten wir zusätzlich den **F1-Score**, das harmonische Mittel aus Präzision (Precision) und Sensitivität (Recall).

Bei mehreren Klassen unterscheiden sich Micro- und Macro-F1: Für Micro-F1 werden alle Beispiele gemeinsam betrachtet. Für Macro-F1 wird der F1-Score für jede Klasse getrennt berechnet und anschließend gemittelt. Somit erhält beim Macro-F1 jede Klasse das gleiche Gewicht; beim Micro-F1 gehen häufige Klassen stärker ein. Im Folgenden implementieren wir Macro-F1 als zusätzliche Bewertungskennzahl.

In [ ]:
@tf.function
def macro_f1_score(y_true, y_pred):
    """
    Calculate the macro F1 score for multi-class classification.
    The macro F1 score is the arithmetic mean of the F1 scores for each class.
    This function assumes that `y_true` and `y_pred` are one-hot encoded.
    Args:
        y_true (tf.Tensor): A tensor of shape (batch_size, num_classes) representing the true labels in one-hot encoded format.
        y_pred (tf.Tensor): A tensor of shape (batch_size, num_classes) representing the predicted labels in one-hot encoded format.
    Returns:
        tf.Tensor: A scalar tensor representing the macro F1 score.
    Example:
        y_true = tf.constant([[1, 0, 0], [0, 1, 0], [0, 0, 1]])
        y_pred = tf.constant([[1, 0, 0], [0, 1, 0], [0, 1, 0]])
        score = macro_f1_score(y_true, y_pred)
        print(score.numpy())  # Output: 0.6666667
    Notes:
        - This function uses TensorFlow operations and is designed to be used within a TensorFlow graph.
        - The function converts one-hot encoded labels to class indices and calculates the F1 score for each class.
        - If a class does not exist in the batch, its F1 score is set to 0.
        - The final macro F1 score is the mean of the F1 scores for all classes.
    """
    
    # One-Hot-Codierung in Klassenindizes umwandeln
    y_true = tf.math.argmax(y_true, axis=-1, output_type=tf.int32)  # Standardize type upfront
    y_pred = tf.math.argmax(y_pred, axis=-1, output_type=tf.int32)

    # Anzahl der Klassen bestimmen
    num_classes = # TODO

    # TensorArray für die F1-Scores initialisieren
    f1_scores = tf.TensorArray(dtype=tf.float32, size=num_classes)

    # Über alle Klassen iterieren
    for i in tf.range(num_classes):
        # Klassenmaske für y_true erzeugen
        class_mask = # TODO

        if tf.reduce_any(class_mask):  # Class exists in batch
            # Richtig positive Vorhersagen zählen
            true_positives = # TODO
            
            # Tatsächlich positive Beispiele zählen
            possible_positives = # TODO
            
            # Als positiv vorhergesagte Beispiele zählen
            predicted_positives = # TODO

            # Präzision und Recall berechnen; epsilon verhindert Division durch 0
            precision = # TODO
            recall = # TODO

            # F1-Score berechnen
            f1_score = # TODO
        else:
            f1_score = 0.0

        f1_scores = f1_scores.write(i, f1_score)

    # Mittleren F1-Score berechnen
    f1_scores = f1_scores.stack()
    return tf.reduce_mean(f1_scores)

Für Transfer Learning laden wir zunächst das vortrainierte `MobileNetV2`. Mit `include_top=False` entfernen wir den ursprünglichen Klassifikationskopf; `weights='imagenet'` lädt die ImageNet-Gewichte. Anschließend ergänzen wir einen passenden Modellkopf: Auf GlobalAveragePooling2D folgt eine Dense-Schicht mit 5 Neuronen und ReLU-Aktivierung. Eine weitere Dense-Schicht besitzt so viele Neuronen wie der Datensatz Klassen und verwendet Softmax.

MobileNetV2 erwartet Bilder mit 224 × 224 Pixeln. Mit `tf.image.resize` passen wir sie an. Eine Lambda-Schicht ermöglicht es, diese Funktion direkt im Modell anzuwenden.

Danach kompilieren wir das Modell mit dem SGD-Optimierer, der vorgegebenen Lernrate und der übergebenen Verlustfunktion (standardmäßig kategorische Kreuzentropie). Als Kennzahlen verwenden wir Accuracy und Macro-F1.

In [ ]:
def build_model(learning_rate, num_unfrozen_layers=0, loss='categorical_crossentropy'):
    # MobileNetV2 mit ImageNet-Gewichten ohne ursprünglichen Klassifikationskopf laden
    base_model = # TODO

    # Basismodell einfrieren
    base_model.trainable = False

    # Letzte Schichten für das Fine-Tuning freigeben
    # TODO

    # Neuen Modellkopf definieren
    new_output = base_model.output
    
    # Add a GlobalAveragePooling2D layer
    new_output = # TODO
    
    # Add a new dense layer with 512 units and ReLU activation
    new_output = # TODO
    
    # Add a new dense layer with 256 units and ReLU activation
    new_output = # TODO

    # Basismodell und neuen Modellkopf verbinden
    modified_base_model = Model(inputs=base_model.input, outputs=new_output)

    # Modell und Skalierungsschicht in ein Sequential-Modell einbetten
    model = K.models.Sequential(
        [
            # Bilder in einer Lambda-Schicht auf die erwartete Größe bringen
            Lambda(
                # TODO: Lambda-Funktion zum Ändern der Bildgröße
                input_shape= # TODO
            ),
            # TODO: Geändertes Basismodell hinzufügen
        ]
    )

    model.summary()

    model.compile(
        optimizer=K.optimizers.SGD(
            learning_rate=learning_rate, momentum=0.9, nesterov=True
        ),
        loss=loss,
        metrics= # TODO: Accuracy und macro_f1_score hinzufügen
    )

    return model

Im ersten Experiment trainieren wir das Modell mit dem ausgewogenen Datensatz. Dieses Ergebnis dient als Basislinie für den Vergleich mit dem unausgewogenen Datensatz.

In [ ]:
model1 = build_model(learning_rate, num_unfrozen_layers)

# Modell trainieren
history1 = model1.fit(
    datagen.flow(x_train, y_train, batch_size=batch_size),
    steps_per_epoch=len(x_train) // batch_size + 1,
    epochs=epochs,
    validation_data=(x_test, y_test),
)

Die Funktion `plot_metrics` aus `util.py` stellt die Kennzahlen für Training und Validierung über die Epochen dar. Zusätzlich zeichnen wir ein Zuverlässigkeitsdiagramm und eine Konfusionsmatrix. Ergänzt dafür die Funktion `plot_diagrams`.

### Zuverlässigkeitsdiagramm
Ein Zuverlässigkeitsdiagramm zeigt die **Kalibrierung** eines Modells: Stimmen vorhergesagte Wahrscheinlichkeiten und tatsächlich beobachtete Trefferhäufigkeiten überein? Die Vorhersagen werden in Wahrscheinlichkeitsintervalle (Bins) eingeteilt. Auf der x-Achse steht die vorhergesagte Konfidenz, auf der y-Achse der beobachtete Anteil korrekter Vorhersagen. `calibration_curve` aus `sklearn.calibration` berechnet die Kurve aus wahren Labels und vorhergesagten Wahrscheinlichkeiten.

### Konfusionsmatrix
Eine Konfusionsmatrix zeigt, wie häufig jede wahre Klasse als eine bestimmte Klasse vorhergesagt wurde. So werden systematische Verwechslungen und schwache Klassen sichtbar, die eine einzelne Accuracy-Zahl verbergen kann. `confusion_matrix` aus `sklearn.metrics` berechnet sie aus wahren und vorhergesagten Labels.

In [ ]:
def plot_diagrams(model, x_test, y_test, label_names, num_bins=5):
    """
    Plots reliability diagrams, entropy vs. confidence, and a confusion matrix.

    Parameters:
    - model: Trained model for predictions.
    - x_test: Test data (numpy array).
    - y_test: True labels (one-hot encoded numpy array).
    - label_names: List of class names.
    - num_bins: Number of bins for calibration curves.
    """
    y_pred = # TODO: Get the predicted probabilities
    predicted_classes = # TODO: Get the predicted classes
    y_true = # Convert one-hot encoded labels to class indices

    fig, axs = plt.subplots(1, 2, figsize=(10, 5))

    # Zuverlässigkeitsdiagramm
    for c, label in enumerate(label_names):
        class_mask = # TODO: Create a mask for the class
        prob_true, prob_pred = # TODO: Calculate true and predicted probabilities using sklearn.calibration.calibration_curve and class_mask
        axs[0].plot(prob_pred, prob_true, marker="o", label=label)
        
    axs[0].plot([0, 1], [0, 1], "k--", label="ideal")
    axs[0].set_title("Zuverlässigkeitsdiagramm")
    axs[0].set_xlabel("Konfidenz")
    axs[0].set_ylabel("Accuracy")
    axs[0].legend()

    # Konfusionsmatrix
    conf_mat = # TODO: Calculate the confusion matrix using sklearn.metrics.confusion_matrix
    im = axs[1].imshow(conf_mat, interpolation="nearest", cmap="Blues")
    axs[1].set_title("Konfusionsmatrix")
    fig.colorbar(im, ax=axs[1])
    tick_marks = np.arange(len(label_names))
    axs[1].set_xticks(tick_marks)
    axs[1].set_xticklabels(label_names, rotation=45)
    axs[1].set_yticks(tick_marks)
    axs[1].set_yticklabels(label_names)

    thresh = conf_mat.max() / 2.0
    for i in range(conf_mat.shape[0]):
        for j in range(conf_mat.shape[1]):
            axs[1].text(
                j,
                i,
                format(conf_mat[i, j], "d"),
                ha="center",
                va="center",
                color="white" if conf_mat[i, j] > thresh else "black",
            )

    axs[1].set_xlabel("Vorhergesagtes Label")
    axs[1].set_ylabel("Wahres Label")
    plt.tight_layout()
    plt.show()

In [ ]:
plot_metrics(history1)
plot_diagrams(model1, x_test, y_test, label_names)

### Fragen:

1. Wie lässt sich das Zuverlässigkeitsdiagramm interpretieren?
2. Welche Verwechslungen zeigt die Konfusionsmatrix?
3. Wie schneidet das Modell gemäß den Kennzahlen ab? Unterscheiden sich Accuracy und Macro-F1?

## Ein Klassenungleichgewicht im Datensatz erzeugen

Nun erzeugen wir ein starkes Klassenungleichgewicht, indem wir Beispiele aus einzelnen Klassen entfernen. Solche Datensätze kommen häufig vor und erschweren das Modelltraining. Werden Messungen in unterschiedlichen Umgebungen erhoben, kann sich die Klassenverteilung von Ort zu Ort verändern. Bei medizinischen Datensätzen kann sie etwa vom Krankenhaus oder von der Erhebungsregion abhängen. Wir untersuchen, wie robust das Modell gegenüber diesem Ungleichgewicht ist und wie gut es auf unbekannte Daten verallgemeinert.

In [ ]:
# Originaldaten für den unausgewogenen Datensatz vorverarbeiten
x_train_imbalanced, y_train_imbalanced = preprocess_data(X, Y)

# Einen unausgewogenen Teildatensatz erzeugen
x_train_imbalanced, y_train_imbalanced = create_imbalanced_subset(
    x_train_imbalanced, y_train_imbalanced, alpha=.1, dataset_length=int(subset_frac * len(X))
)

# Testdaten bleiben unverändert
x_test_imbalanced = x_test
y_test_imbalanced = y_test

# Datengenerator für die Datenerweiterung erstellen
datagen_imbalanced = data_augmentation(x_train_imbalanced)

# Klassenverteilung des unausgewogenen Trainingssatzes darstellen
plot_label_distribution(y_train_imbalanced)
plot_label_distribution(y_test_imbalanced)

print("Anzahl Trainingsbeispiele:", len(x_train_imbalanced))
print("Anzahl Testbeispiele:", len(x_test_imbalanced))

## MobileNetV2 mit dem unausgewogenen Datensatz trainieren

Wir trainieren dasselbe Modell mit dem unausgewogenen Datensatz und bewerten es anhand von Accuracy und F1-Score. Anschließend vergleichen wir beide Trainingsfälle. Auch hier zeichnen wir Zuverlässigkeitsdiagramm und Konfusionsmatrix.

In [ ]:
model2 = build_model(learning_rate, num_unfrozen_layers) 

# Modell trainieren
history2 = model2.fit(
    datagen_imbalanced.flow(x_train_imbalanced, y_train_imbalanced, batch_size=batch_size),
    steps_per_epoch=len(x_train_imbalanced) // batch_size + 1,
    epochs=epochs,
    validation_data=(x_test_imbalanced, y_test_imbalanced),
)

In [ ]:
plot_metrics(history2)
plot_diagrams(model2, x_test_imbalanced, y_test_imbalanced, label_names)

### Fragen:

1. Wie unterscheiden sich die Kennzahlen im ausgewogenen und unausgewogenen Fall?
2. Wie lassen sich die Veränderungen in Zuverlässigkeitsdiagramm und Konfusionsmatrix erklären?
3. Unterscheiden sich Accuracy und Macro-F1? Welche Kennzahl beschreibt das Problem besser?

## Klassen-Gewichte zum Umgang mit dem Ungleichgewicht verwenden

Eine einfache Methode sind **Klassen-Gewichte** in der Verlustfunktion. Seltene Klassen erhalten ein höheres, häufige Klassen ein niedrigeres Gewicht. Fehler bei seltenen Klassen beeinflussen das Training dadurch stärker. Das Argument `class_weight` von `model.fit` übergibt die Gewichte. In `compute_class_weight` berechnen wir sie aus der Gesamtzahl der Beispiele und der Häufigkeit jeder Klasse.

In [ ]:
def compute_class_weight(Y):
    # One-Hot-Codierung bei Bedarf in Klassenindizes umwandeln
    if Y.shape[1] > 1:
        Y = np.argmax(Y, axis=1)

    # Klassen-Gewichte berechnen
    # TODO

    return class_weights

# Klassen-Gewichte berechnen
class_weights = compute_class_weight(y_train_imbalanced)
class_weights = {i: class_weights[i] for i in range(len(class_weights))}

model3 = build_model(learning_rate, num_unfrozen_layers) 

# Modell trainieren
history3 = model3.fit(
    datagen_imbalanced.flow(x_train_imbalanced, y_train_imbalanced, batch_size=batch_size),
    steps_per_epoch=len(x_train_imbalanced) // batch_size + 1,
    epochs=epochs,
    validation_data=(x_test_imbalanced, y_test_imbalanced),
    class_weight=class_weights,
)

In [ ]:
# Kennzahlen darstellen
plot_metrics(history3)
plot_diagrams(model3, x_test_imbalanced, y_test_imbalanced, label_names)

### Fragen

1. Wie helfen Klassen-Gewichte bei einem Klassenungleichgewicht?
2. Wie verändern sie die gemessenen Kennzahlen?
3. Welche Veränderungen zeigen Zuverlässigkeitsdiagramm und Konfusionsmatrix?

## Focal Loss zum Umgang mit dem Ungleichgewicht verwenden

Eine weitere Methode ist **Focal Loss**, eine Variante der Kreuzentropie. Schwer zu klassifizierende Beispiele erhalten mehr Gewicht, leicht zu klassifizierende weniger. Das Modell soll sich dadurch stärker auf schwierige – häufig auch seltene – Beispiele konzentrieren:

$$
\text{Focal Loss} = -\sum_{i=1}^{N} \alpha_i (1 - p_i)^\gamma \log(p_i)
$$

Dabei ist $N$ die Anzahl der Klassen, $\alpha_i$ das Gewicht der Klasse $i$, $p_i$ die vorhergesagte Wahrscheinlichkeit und $\gamma$ der Fokussierungsparameter. Wir definieren Focal Loss als eigene Verlustfunktion, trainieren damit und bewerten das Modell auf dem unausgewogenen Datensatz.

Für $\alpha$ verwenden wir die zuvor berechneten Klassen-Gewichte; $\gamma$ setzen wir auf 2.

In [ ]:
def focal_loss(gamma=2.0, alpha=None):
    def focal_loss_fixed(y_true, y_pred):
        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)
        y_true = tf.cast(y_true, tf.float32)
        
        if alpha is not None:
            alpha_t = # TODO: Calculate alpha_t
        else:
            alpha_t = tf.ones_like(y_true)
            
        p_t = # TODO: Calculate p_t
        fl = # TODO: Calculate focal loss
        
        return tf.reduce_sum(fl, axis=-1)

    return focal_loss_fixed

alpha = # TODO: Set the alpha value to class weights like in the previous example
model4 = # TODO: Build the model with focal loss

# Modell trainieren with early stopping, model checkpointing, and data augmentation
history4 = model4.fit(
    datagen_imbalanced.flow(x_train_imbalanced, y_train_imbalanced, batch_size=batch_size),
    steps_per_epoch=len(x_train_imbalanced) // batch_size + 1,
    epochs=epochs,
    validation_data=(x_test_imbalanced, y_test_imbalanced),
)

In [ ]:
# Kennzahlen darstellen
plot_metrics(history4)
plot_diagrams(model4, x_test_imbalanced, y_test_imbalanced, label_names)

### Fragen

1. Wie kann Focal Loss bei einem Klassenungleichgewicht helfen?
2. Wie verändert Focal Loss die gemessenen Kennzahlen?
3. Welche Veränderungen zeigen Zuverlässigkeitsdiagramm und Konfusionsmatrix?

## Qualitative Analyse mit Grad-CAM

Neben den Zahlen möchten wir verstehen, worauf ein Modell seine Vorhersagen stützt. Deep-Learning-Modelle verhalten sich häufig wie eine Blackbox. Methoden der erklärbaren künstlichen Intelligenz (Explainable AI, XAI) sollen Hinweise auf das Modellverhalten geben. Grad-CAM (Gradient-weighted Class Activation Mapping) erzeugt aus den Gradienten der Zielklasse bezüglich der letzten Faltungsschicht eine Heatmap. Sie markiert Bildbereiche, die die Vorhersage stark beeinflusst haben.

`gradcam_analysis` aus `util.py` zeigt das Originalbild neben einer Kopie mit überlagerter Grad-CAM-Heatmap.

Zunächst implementieren wir `get_model_predictions`. Die Funktion gibt die größte vorhergesagte Wahrscheinlichkeit (Konfidenz), die vorhergesagte Klasse und einen Indikator für eine korrekte Vorhersage zurück. Wir berechnen diese Werte für alle vier Modelle.

> **Achtung:** Eine Grad-CAM-Heatmap ist eine Interpretationshilfe. Sie beweist weder Ursache noch medizinische Relevanz eines hervorgehobenen Bereichs.

In [ ]:
def get_model_predictions(model, x_test, y_test):
    # y_test bei Bedarf in Klassenindizes umwandeln
    if len(y_test.shape) > 1:
        y_test = np.argmax(y_test, axis=1)

    # Modellvorhersagen berechnen
    y_pred = # TODO: Get the model predictions

    # Vorhergesagte Klassen bestimmen
    y_pred_classes = # TODO: Get the class predictions

    # Klassenwahrscheinlichkeiten bestimmen
    y_pred_prob = # TODO: Get the class probabilities

    # Indikator für korrekte Vorhersagen bestimmen
    y_pred_correct = # TODO: Get the indicators for correct class predictions

    return {
        "y_pred_prob": y_pred_prob,
        "y_pred_classes": y_pred_classes,
        "y_pred_correct": y_pred_correct,
    }


model_predictions = [
    get_model_predictions(model, x_test_imbalanced, y_test_imbalanced)
    for model in [model1, model2, model3, model4]
]

true_classes = np.argmax(y_test_imbalanced, axis=1)

Nun filtern wir besonders interessante Beispiele. Wir unterscheiden zwei Fälle:

1. Modell 2 klassifiziert ein Beispiel falsch – möglicherweise infolge des Klassenungleichgewichts –, alle anderen Modelle klassifizieren es richtig.
2. Alle Modelle klassifizieren ein Beispiel richtig, ihre Konfidenzen unterscheiden sich jedoch deutlich.

Die zugehörigen Indizes speichern wir in `sample_idxs_1` und `sample_idxs_2` für die spätere Grad-CAM-Analyse.

In [ ]:
# Fall 1: Modell 2 klassifiziert falsch, alle anderen richtig
# TODO

# Fall 2: Alle klassifizieren richtig, aber die Konfidenzen unterscheiden sich stark
# TODO

Die Funktion `plot_grad_cam` erhält den Index eines Beispiels und ruft `gradcam_analysis` für jedes zuvor trainierte Modell auf.

Verwendet die Funktion, um verschiedene Beispiele aus beiden Fällen zu untersuchen.

In [ ]:
def plot_grad_cam(sample_idx):
    gradcam_analysis(
        x_test_imbalanced[sample_idx],
        model1.layers[1],
        "Conv_1",
        model_predictions[0]["y_pred_classes"][sample_idx],
        true_classes[sample_idx],
        model_predictions[0]["y_pred_prob"][sample_idx],
        label_names,
        title="Model 1",
    )

    gradcam_analysis(
        x_test_imbalanced[sample_idx],
        model2.layers[1],
        "Conv_1",
        model_predictions[1]["y_pred_classes"][sample_idx],
        true_classes[sample_idx],
        model_predictions[1]["y_pred_prob"][sample_idx],
        label_names,
        title="Model 2",
    )

    gradcam_analysis(
        x_test_imbalanced[sample_idx],
        model3.layers[1],
        "Conv_1",
        model_predictions[2]["y_pred_classes"][sample_idx],
        true_classes[sample_idx],
        model_predictions[2]["y_pred_prob"][sample_idx],
        label_names,
        title="Model 3",
    )

    gradcam_analysis(
        x_test_imbalanced[sample_idx],
        model4.layers[1],
        "Conv_1",
        model_predictions[3]["y_pred_classes"][sample_idx],
        true_classes[sample_idx],
        model_predictions[3]["y_pred_prob"][sample_idx],
        label_names,
        title="Model 4",
    )

In [ ]:
plot_grad_cam(sample_idxs_1[0])

In [ ]:
plot_grad_cam(sample_idxs_2[-1])

### Fragen:

1. Welche Hinweise liefert die Grad-CAM-Analyse – und was kann sie nicht beweisen?
2. Wie unterscheiden sich Vorhersagen und hervorgehobene Bereiche der Modelle?